# Phase 3b — Shifts that can raise the evidence for absent objects

Tests whether hallucination filters calibrated on clean COCO + LLaVA + OWLv2 stay valid when:
- **A. the detector is swapped** (OWL-ViT v1, Grounding DINO), on the same 200 pilot images;
- **B. the VLM is swapped** (Qwen2.5-VL-3B), on the same 200 pilot images;
- **C. the images come from another domain** (200 AMBER images, non-COCO).

**Kaggle settings:** Accelerator **GPU T4 x2**, **Internet ON**.

**How to run (2 steps):**
1. Run cells 1–4 interactively (smoke test, ~10–15 min). If cell 4 prints captions and scores without errors, continue.
2. Click **Save Version → Save & Run All (Commit)**. The full run (~3–4 h) continues in the background, even if you close the browser. When it finishes, open the version's **Output** tab and download `BCRBP_phase3b/phase3b_results.zip`.

Every stage resumes where it stopped, so re-running is always safe.

In [ ]:
# 1. Environment and GPU
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
print("Kaggle:", ON_KAGGLE, "| Colab:", ON_COLAB)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
      or "NO GPU FOUND - switch the accelerator to GPU T4 x2")

In [ ]:
# 2. Output folder
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, WORK = "/content/drive/MyDrive/BCRBP_phase3b", "/content"
elif ON_KAGGLE:
    OUT, WORK = "/kaggle/working/BCRBP_phase3b", "/kaggle/working"
else:
    OUT, WORK = os.path.abspath("BCRBP_phase3b"), os.getcwd()
os.makedirs(OUT, exist_ok=True)
print("Results go to:", OUT)

In [ ]:
# 3. Code (branch phase3b-shifts) and packages
REPO = "https://github.com/antonyjasfer/Budget-Coupled-Robust-Belief-Propagation-for-Visual-Hallucination-Detection.git"
BRANCH = "phase3b-shifts"
os.chdir(WORK)
if not os.path.exists("bcrbp"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "bcrbp"], check=True)
else:
    subprocess.run(["git", "-C", "bcrbp", "pull", "--ff-only"], check=True)
os.chdir(os.path.join(WORK, "bcrbp"))
!pip -q install inflect bitsandbytes accelerate gdown
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())
!git log --oneline -1

In [ ]:
# 4. Smoke test: a few images per arm (~10-15 min). Fix any error here before the full run.
!python scripts/phase3b.py setup --out "$OUT"
!python scripts/phase3b.py swap  --out "$OUT" --limit 4
!python scripts/phase3b.py qwen  --out "$OUT" --limit 2
!python scripts/phase3b.py amber --out "$OUT" --limit 2
import json
show = lambda f: [json.loads(l) for l in open(f"{OUT}/{f}")] if os.path.exists(f"{OUT}/{f}") else []
print("swap:", show("swap_evidence.jsonl")[:1])
print("\nQwen caption:", (show("qwen_captions.jsonl") or [{"caption": "MISSING"}])[0]["caption"][:400])
print("\nAMBER caption:", (show("amber_captions.jsonl") or [{"caption": "MISSING"}])[0]["caption"][:400])

In [ ]:
# 5. Full run (~3-4 h). Safe to re-run after a disconnect.
# If Qwen runs out of GPU memory, add --qwen-4bit to the qwen line.
!python scripts/phase3b.py swap  --out "$OUT"
!python scripts/phase3b.py qwen  --out "$OUT"
!python scripts/phase3b.py amber --out "$OUT"

In [ ]:
# 6. Analysis (CPU, a few minutes) and report
!python scripts/phase3b.py analyze --out "$OUT"
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/phase3b_report.md").read()))

In [ ]:
# 7. Package results (no images) for sharing
import zipfile
keep = ["swap_evidence.jsonl", "qwen_captions.jsonl", "qwen_evidence.jsonl", "amber/selected_ids.json",
        "amber_captions.jsonl", "amber_evidence.jsonl", "amber_claims.jsonl",
        "phase3b_results.json", "phase3b_report.md"]
zpath = os.path.join(OUT, "phase3b_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in keep:
        p = os.path.join(OUT, f)
        if os.path.exists(p):
            z.write(p, f)
    z.writestr("environment.txt", f"torch {torch.__version__}\ntransformers {transformers.__version__}\n"
               + subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("Saved:", zpath, round(os.path.getsize(zpath) / 1e6, 1), "MB")
if ON_KAGGLE:
    print("Kaggle: Output tab -> BCRBP_phase3b/phase3b_results.zip")